braincell version: 0.0.8
repo_root: D:\codes\projects\braincell


# `io`: query NeuroMorpho metadata, download `SWC`, then load with `braincell`

This notebook demonstrates the smallest practical NeuroMorpho workflow in code:

1. Query the public REST API with a simple filter
2. Preview the matching neuron records
3. Download one standardized `SWC` file
4. Import the downloaded file with `Morpho.from_swc(...)`

The example keeps the filter intentionally small: one main query field plus one extra facet filter.


## 1. Query NeuroMorpho with `q` and `fq`

NeuroMorpho exposes neuron metadata from `https://neuromorpho.org/api/neuron/select/`.

In practice, the easiest pattern is:

- `q="species:mouse"` for the main query
- `fq=["brain_region:neocortex"]` for extra filters

You can discover supported field names from `https://neuromorpho.org/api/neuron/fields`.


In [2]:
API_BASE = "https://neuromorpho.org/api"
FILE_BASE = "https://neuromorpho.org/dableFiles"
session = requests.Session()


def fetch_neurons(*, query, filters=None, size=5, page=0, sort="neuron_id,asc"):
    params = {
        "q": query,
        "size": size,
        "page": page,
        "sort": sort,
    }
    if filters:
        params["fq"] = [f"{key}:{value}" for key, value in filters.items()]

    response = session.get(f"{API_BASE}/neuron/select/", params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    neurons = payload.get("_embedded", {}).get("neuronResources", [])
    return neurons, payload.get("page", {}), response.url


def _as_text(value):
    if isinstance(value, list):
        return "; ".join(str(x) for x in value)
    return value


def neurons_to_df(neurons):
    rows = []
    for neuron in neurons:
        rows.append(
            {
                "neuron_id": neuron["neuron_id"],
                "neuron_name": neuron["neuron_name"],
                "archive": neuron["archive"],
                "species": neuron["species"],
                "brain_region": _as_text(neuron.get("brain_region")),
                "cell_type": _as_text(neuron.get("cell_type")),
                "experiment_condition": _as_text(neuron.get("experiment_condition")),
            }
        )
    return pd.DataFrame(rows)


def measurement_url(neuron):
    return neuron["_links"]["measurements"]["href"]


def fetch_measurement(neuron):
    response = session.get(measurement_url(neuron), timeout=30)
    response.raise_for_status()
    return response.json()


def measurement_to_df(measurement):
    preferred_keys = ("neuron_id", "neuron_name")
    ordered_keys = [key for key in preferred_keys if key in measurement]
    ordered_keys.extend(sorted(key for key in measurement if key not in preferred_keys))
    return dict_to_df({key: measurement[key] for key in ordered_keys})


def compare_measurement_keys(neurons, *, limit=5):
    sampled_neurons = neurons[:limit]
    measurements = [fetch_measurement(neuron) for neuron in sampled_neurons]
    if not measurements:
        return pd.DataFrame(), tuple()

    reference_keys = tuple(sorted(measurements[0].keys()))
    rows = []
    for neuron, measurement in zip(sampled_neurons, measurements):
        keys = tuple(sorted(measurement.keys()))
        rows.append(
            {
                "neuron_id": neuron["neuron_id"],
                "neuron_name": neuron["neuron_name"],
                "same_as_reference": keys == reference_keys,
                "key_count": len(keys),
                "missing_keys": tuple(sorted(set(reference_keys) - set(keys))),
                "extra_keys": tuple(sorted(set(keys) - set(reference_keys))),
            }
        )
    return pd.DataFrame(rows), reference_keys


def standardized_swc_url(neuron):
    archive = quote(neuron["archive"].lower(), safe="")
    neuron_name = quote(neuron["neuron_name"], safe="")
    return f"{FILE_BASE}/{archive}/CNG%20version/{neuron_name}.CNG.swc"


def safe_filename(name):
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return cleaned.strip("._") or "neuromorpho_neuron"


def download_swc(neuron, output_dir, *, overwrite=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = f"{safe_filename(neuron['neuron_name'])}.CNG.swc"
    output_path = output_dir / filename
    if output_path.exists() and not overwrite:
        return output_path, standardized_swc_url(neuron), False

    url = standardized_swc_url(neuron)
    with session.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with output_path.open("wb") as fh:
            for chunk in response.iter_content(chunk_size=1 << 15):
                if chunk:
                    fh.write(chunk)
    return output_path, url, True


,key,value
0,query,species:mouse
1,filters,{'brain_region': 'neocortex'}
2,request_url,https://neuromorpho.org/api/neuron/select/?q=s...
3,total_matches,54580
4,page_size,5


In [4]:
neurons_to_df(neurons)

,neuron_id,neuron_name,archive,species,brain_region,cell_type,experiment_condition
0,10047,TypeA-10,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
1,10048,TypeB-06,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
2,10049,ChR2-negative-03,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
3,10050,ChR2-negative-06,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
4,10051,TypeA-09,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control


In [ ]:
query = "species:mouse"
filters = {"brain_region": "neocortex"}

neurons, page_info, query_url = fetch_neurons(
    query=query,
    filters=filters,
    size=5,
)
if not neurons:
    raise RuntimeError("No NeuroMorpho records matched the current filters. Relax the filters and run again.")

display(
    dict_to_df(
        {
            "query": query,
            "filters": filters,
            "request_url": query_url,
            "total_matches": page_info.get("totalElements"),
            "page_size": page_info.get("size"),
        }
    )
)
display(neurons_to_df(neurons))

selected_neuron = neurons[0]
selected_measurement_url = measurement_url(selected_neuron)
display(
    dict_to_df(
        {
            "selected_neuron": selected_neuron["neuron_name"],
            "selected_neuron_id": selected_neuron["neuron_id"],
            "archive": selected_neuron["archive"],
            "download_url": standardized_swc_url(selected_neuron),
            "measurement_url": selected_measurement_url,
        }
    )
)


,key,value
0,selected_neuron,TypeA-10
1,selected_neuron_id,10047
2,archive,Scanziani
3,download_url,https://neuromorpho.org/dableFiles/scanziani/C...
4,measurement_url,http://neuromorpho.org/api/morphometry/id/10047


## 2. Inspect NeuroMorpho measurements

Each neuron metadata record links to a NeuroMorpho morphometry endpoint under `_links.measurements.href`.

NeuroMorpho reports these morphometrics from **L-Measure**. Two official rules matter before you try to map them to your own metrics:

- Except for `Soma_Surface` and `N_stems`, the reported metrics exclude compartments defined as soma.
- `width`, `height`, and `depth` are **not** raw `x/y/z` extents. NeuroMorpho first translates the soma to the origin, orients the whole neuron by PCA/SVD-like principal axes, and then reports the minimum span containing 95% of tracing points along those axes.

| field | meaning | includes soma? | scope / aggregation | how it is computed | notes for later mapping |
| --- | --- | --- | --- | --- | --- |
| `neuron_id` | NeuroMorpho record id | n/a | identifier | database id, not a morphometric calculation | use only as a join key |
| `neuron_name` | NeuroMorpho reconstruction name | n/a | identifier | dataset-provided name, not a morphometric calculation | use only as a join key |
| `soma_Surface` | soma surface area | yes, soma only | single soma summary | computed from NeuroMorpho's standardized soma representation | this is separate from `surface`; do not add it again if another metric already includes soma |
| `n_stems` | number of stems / trees emerging from the soma | yes, soma-connected context | count of primary neurites | official FAQ describes this as total number of trees; practically this is the number of primary branches attached to the soma | closest semantic match is usually soma children, not total branch count |
| `n_bifs` | number of bifurcation points | no | whole-neuron total | total count of branch points in neurites | should usually align with a branch-point count that excludes soma |
| `n_branch` | number of neuritic branches | no | whole-neuron total | official FAQ: total number of branches = bifurcations plus terminations; official update: soma excluded | if your tree model stores soma/root as a branch, expect an off-by-one vs. `braincell.n_branches` |
| `width` | neuronal width | no in the reported span; orientation still uses whole neuron | whole-neuron global extent | 95% span along the **second** principal component after translating soma to origin and orienting the whole neuron by PCA/SVD-like major axes | this is not the same as raw `x_range` |
| `height` | neuronal height | no in the reported span; orientation still uses whole neuron | whole-neuron global extent | 95% span along the **first** principal component after soma-centered PCA/SVD-like orientation | this is not the same as raw `y_range` unless axes already align |
| `depth` | neuronal depth | no in the reported span; orientation still uses whole neuron | whole-neuron global extent | 95% span along the **third** principal component after soma-centered PCA/SVD-like orientation | this is not raw `z_max - z_min`; it is the thinnest PCA axis |
| `diameter` | average branch diameter | no | mean over neuritic branches/compartments | official FAQ: average branch diameter | compare only to metrics that average neuritic diameters and exclude soma |
| `length` | total arborization length | no | whole-neuron total | sum of neuritic path lengths | closest semantic match is neurite-only total length |
| `surface` | total arborization surface area | no | whole-neuron total | official FAQ: total arborization surface area | this is **not** soma + neurite surface; `soma_Surface` is separate |
| `volume` | total internal volume of the arborization | no | whole-neuron total | official FAQ: total internal volume of the arborization | compare to neurite-only volume, not soma-inclusive volume |
| `eucDistance` | maximum Euclidean distance from soma to tips | soma is the reference point, but soma compartments are excluded from the measured arbor | tip-level maximum summarized to one value | maximum straight-line distance from soma to terminal tips | compare to a soma-to-tip Euclidean max, not branch-local Euclidean length |
| `pathDistance` | maximum path distance from soma to tips | soma is the reference point, but soma compartments are excluded from the measured arbor | tip-level maximum summarized to one value | maximum along-tree distance from soma to terminal tips | compare to a soma-to-tip geodesic/path max |
| `branch_Order` | maximum branch order | no | tip-level maximum summarized to one value | number of bifurcations from soma to tips, then take the maximum | useful to compare with a centrifugal branch-order definition |
| `contraction` | average contraction | no | average over branches | ratio between Euclidean length and path length calculated on each branch, then averaged | lower values mean more tortuous branches |
| `fragmentation` | total number of reconstruction points | no | whole-neuron total | official FAQ: total number of reconstruction points; 2013 update: stem branches are included | this is a tracing-resolution count, not a topological count |
| `partition_asymmetry` | topological asymmetry | no | average over bifurcations | average of `abs(n1 - n2) / (n1 + n2 - 2)`, where `n1` and `n2` are the numbers of terminal tips in the two daughter subtrees | compare only with the same subtree-tip definition |
| `pk_classic` | Average Rall's Ratio | no | average over bifurcations | average over bifurcations of `(d1^1.5 + d2^1.5) / b^1.5`; NeuroMorpho detail pages label this as Average Rall's Ratio | API key is `pk_classic`, but the UI label is Rall's Ratio |
| `bif_ampl_local` | local bifurcation angle | no | average over bifurcations | average angle between the first two daughter compartments | compare only with a local daughter-segment angle, not a downstream endpoint angle |
| `bif_ampl_remote` | remote bifurcation angle | no | average over bifurcations | average angle between the following bifurcations or tips; 2013 update says terminal branches are included | this is a downstream geometry measure, not the immediate daughter angle |
| `fractal_Dim` | fractal dimension | no | branchwise calculation summarized to one neuron-level value | official update: for each branch, compute the slope of the regression line between `log10(path distance)` and `log10(Euclidean distance)` of tracing points measured from the branch origin; NeuroMorpho then reports one summary value for the neuron | the branch-level formula is official; the exact neuron-level aggregation is not spelled out on the update page |

Because NeuroMorpho uses L-Measure conventions, these values will not always match `braincell` summaries one-to-one. The biggest causes are soma inclusion rules, PCA-based width/height/depth, and whether your own tree model counts the soma/root as a branch.

Sources: NeuroMorpho official morphometric updates (24 June 2013), NeuroMorpho FAQ entries on pre-computed morphometrics and orientation, and NeuroMorpho detail-page labels such as "Average Rall's Ratio".


,key,value
0,measurement_check_limit,5
1,reference_key_count,23
2,reference_keys,"(bif_ampl_local, bif_ampl_remote, branch_Order..."


,neuron_id,neuron_name,same_as_reference,key_count,missing_keys,extra_keys
0,10047,TypeA-10,True,23,(),()
1,10048,TypeB-06,True,23,(),()
2,10049,ChR2-negative-03,True,23,(),()
3,10050,ChR2-negative-06,True,23,(),()
4,10051,TypeA-09,True,23,(),()


In [ ]:
measurement_check_df, reference_measurement_keys = compare_measurement_keys(neurons, limit=5)
selected_measurement = fetch_measurement(selected_neuron)

display(
    dict_to_df(
        {
            "measurement_check_limit": min(5, len(neurons)),
            "reference_key_count": len(reference_measurement_keys),
            "reference_keys": reference_measurement_keys,
        }
    )
)
display(measurement_check_df)
display(measurement_to_df(selected_measurement))


,key,value
0,neuron_id,10047
1,neuron_name,TypeA-10
2,bif_ampl_local,71.93
3,bif_ampl_remote,75.51
4,branch_Order,8.0
5,contraction,0.79
6,depth,23.89
7,diameter,0.11
8,eucDistance,542.71
9,fractal_Dim,1.03


## 3. Download one standardized `SWC`

NeuroMorpho exposes normalized files under a predictable URL pattern:

- `https://neuromorpho.org/dableFiles/{archive}/CNG version/{neuron_name}.CNG.swc`

The helper above builds this URL from the selected metadata row and saves the file locally.


,key,value
0,download_url,https://neuromorpho.org/dableFiles/scanziani/C...
1,saved_to,/home/swl/braincell/develop_doc/data/neuromorp...
2,downloaded_now,False
3,file_size_bytes,80865


In [ ]:
output_dir = repo_root / "develop_doc" / "data" / "neuromorpho"
downloaded_path, download_url, downloaded_now = download_swc(
    selected_neuron,
    output_dir,
    overwrite=False,
)

display(
    dict_to_df(
        {
            "download_url": download_url,
            "saved_to": downloaded_path,
            "downloaded_now": downloaded_now,
            "file_size_bytes": downloaded_path.stat().st_size,
        }
    )
)

header_preview = downloaded_path.read_text(errors="replace").splitlines()[:5]
print("first lines:")
print("\n".join(header_preview))


first lines:
# Original file TypeA-10.swc edited using StdSwc version 1.31 on 5/5/14.
# Irregularities and fixes documented in TypeA-10.swc.std.  See StdSwc1.31.doc for more information.
#
# Neurolucida to SWC conversion from L-Measure. Sridevi Polavaram: spolavar@gmu.edu
# Original fileName:C:\Users\praveen\Desktop\DataProcessing\CurrentArchives\Processing\Olsen-Scanziani\ASC\TypeA-10.asc


## 4. Load the downloaded file with `braincell`

Once the `SWC` file is on disk, the rest of the workflow is the same as any local import:

- `tree = Morpho.from_swc(path)`
- `tree, report = Morpho.from_swc(path, return_report=True)`


In [ ]:
tree, report = Morpho.from_swc(downloaded_path, return_report=True)

display(morpho_summary_df(tree))
display(morpho_report_summary_df(report))
display(morpho_report_df(report).head(10))
print("topology preview:")
print("\n".join(tree.topo().splitlines()[:8]))


114.63587 * umetre

## 5. Next adjustments

To adapt this example for your own download script, the main knobs are:

- change `query`, for example `"species:rat"`
- add or remove entries in `filters`, for example `{"brain_region": "hippocampus"}`
- increase `size` if you want to preview more matches before choosing one
- use `_links.measurements.href` if you want the NeuroMorpho morphometry payload for each neuron
- set `overwrite=True` in `download_swc(...)` if you want to refresh a local copy

If you need the full list of filterable fields, call `GET https://neuromorpho.org/api/neuron/fields` and reuse the same `field:value` syntax shown here.
